# Dự án 4 — Cuộc thi chọn model — Đề chung: Hạt đậu khô + đề chọn thêm
## BẢN CỦA NHÓM

**Trường THCS và THPT Đinh Thiện Lý — Machine Learning Level 1 — Khối 10**

---

**Cuộc thi:** máy chụp ảnh từng hạt đậu khô và đo 16 con số về hình dạng (diện tích, chu vi, độ tròn…). Nhóm dạy máy phân biệt **7 loại hạt**. Bảng có 10 888 hạt có nhãn; thầy giữ kín nhãn của 2 723 hạt khác — đó là **đề thi** chấm ở tuần 4, xếp hạng cả lớp.
Tuần 3 nhóm chọn thêm **một** bộ thứ hai (nấm hoặc Pokémon) để kiểm tra: model thắng ở hạt đậu có thắng ở mọi nơi không?
**Đề có hướng dẫn** (nhóm cần nhiều khung hơn): notebook Bài 19 (động vật) hoặc Bài 23 (nước uống), rồi vẫn nộp dự đoán hạt đậu ở tuần 4.

**Câu hỏi lớn của dự án:** *Model nào tốt nhất — và “tốt nhất” nghĩa là gì?*

**Dữ liệu:** Dry Bean — Koklu & Ozkan (2020), CC BY 4.0 · Mushroom (UCI), CC BY 4.0 · Pokémon (Kaggle), CC0  

> Bảng xếp hạng chỉ để vui và để học — **không tính điểm**. Rubric mới tính điểm. Bộ nấm là mẫu giả định theo sách hướng dẫn: **không bao giờ dùng model để quyết định ăn nấm thật** — nấm độc có thể gây chết người.

### Các mốc (nộp trên Canvas trước 23:59 Thứ 7 của tuần đó)

| Tuần | Mốc | Nộp |
|---|---|---|
| 1 · buổi 18 | **M1 + M2** Mốc + cây quyết định; đọc cây | Notebook bước 1 – 2 |
| 2 · buổi 19 | **M3a** Bảng so sánh 6 model | Notebook bước 3 |
| 3 · buổi 20 | **M3b** Kiểm định chéo, dò siêu tham số, đề chọn thêm | Notebook bước 4, 4b |
| 4 · buổi 21 | **M4** Nộp dự đoán đề thi, 1 slide “vì sao chọn model này” | File dự đoán + slide + Nhật ký AI |

### Nhóm và vai — **đổi vai mỗi tuần**

Nhóm: … · Lớp: …

| Vai | Tuần 1 | Tuần 2 | Tuần 3 | Tuần 4 | Việc chính |
|---|---|---|---|---|---|
| Lái code | … | … | … | … | Gõ và chạy notebook; không ai gõ hộ hai tuần liền |
| Nghiên cứu | … | … | … | … | Đọc bài thư viện, hỏi Gemini “vì sao”, ghi Nhật ký AI |
| Thư ký số liệu | … | … | … | … | Ghi bảng thí nghiệm, kiểm tra con số trước khi báo cáo |
| Trình bày | … | … | … | … | Báo cáo 1 phút mỗi tuần, làm slide / poster |

### Quy tắc dùng AI

- **Được:** hỏi Gemini (✨ trong Colab) một lệnh làm gì, bấm *Giải thích lỗi*, hỏi *vì sao* làm bước này, nhờ gợi ý cách tiếp cận.
- **Không được:** nộp con số, biểu đồ, kết luận mà nhóm không tự chạy ra; để AI viết cả notebook.
- Ô code do AI gợi ý ghi dòng đầu `# AI gợi ý — nhóm đã kiểm tra: ...`. Mỗi tuần ghi **Nhật ký AI** ở cuối.
- Khi đi qua nhóm và ở mốc cuối, thầy gọi **một bạn bất kỳ** giải thích một dòng code hoặc một con số.

### Dữ liệu

| Cột | Ý nghĩa |
|---|---|
| `id` | Số thứ tự hạt (giữ khi nộp) |
| `Area, Perimeter` | Diện tích, chu vi (đơn vị điểm ảnh) |
| `MajorAxisLength, MinorAxisLength` | Trục dài, trục ngắn |
| `roundness, Compactness` | Độ tròn, độ chắc |
| `ShapeFactor1 – 4, …` | Các chỉ số hình dạng khác |
| `loai` | Nhãn: 7 loại hạt (tên tiếng Thổ Nhĩ Kỳ) |


---
# TUẦN 1 (buổi 18) — Mốc và cây quyết định

**Mốc của tuần:** **M1 + M2** Mốc + cây quyết định; đọc cây  
**Học trước (bài thư viện trên web, 3 chặng bắt buộc + checkpoint):** [Bài 18 Cây quyết định](https://vuongndlst.github.io/ml-level1/bai18/)


---
## Bước 0 — Nạp dữ liệu (đã viết sẵn, chỉ cần chạy)


In [ ]:
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
URL = "https://vuongndlst.github.io/ml-level1/du-lieu/"
def doc(tep):   # đã tải file lên Colab thì đọc file; chưa thì đọc thẳng từ web khoá học
    return pd.read_csv(tep if os.path.exists(tep) else URL + tep)
df = doc("hat_dau.csv")
de_thi = doc("hat_dau_de_thi.csv")   # KHÔNG có nhãn — để nộp thi
print(df.shape, de_thi.shape)
df.head()


### Câu hỏi của nhóm (M1)

*Gợi ý câu hỏi:* Loại hạt nào dễ nhầm nhất? · Cây hỏi câu gì đầu tiên? · Model nào sẽ thắng?

| | Nhóm điền |
|---|---|
| Câu hỏi nhóm chọn | … |
| Cột sẽ dùng | … |
| Dự đoán trước (giả thuyết) | … |
| Làm sao biết dự đoán đúng hay sai? | … |


---
## Bước 1 — Khám phá

> **🎯 Mục tiêu:** Biết 7 lớp có cân bằng không, thang đo các cột ra sao.
>
> **🧭 Vì sao bước này?** Chưa hiểu mỗi dòng, mỗi cột là gì thì không biết hỏi dữ liệu câu gì, cũng không biết cột nào là đáp án (nhãn) và cột nào dùng để đoán (feature).
>
> **📋 Nhiệm vụ:**
> 1. Đếm số hạt mỗi loại.
> 2. Đếm dòng trùng.
> 3. Xem min, max từng cột.
>
> **💡 Gợi ý lệnh:**
> - `df["loai"].value_counts()` — đếm mỗi lớp
> - `df.describe().T` — thang đo
>
> **✅ Đạt khi:** nói được lớp nhiều nhất, lớp ít nhất, cột có thang đo lớn nhất.


In [ ]:
# Buoc 1 - code cua nhom



**Nhận xét của nhóm:** …


---
## Bước 2 — Mốc và cây đầu tiên

> **🎯 Mục tiêu:** Chia train / validation 75 / 25; mốc; cây sâu 3 tầng; đọc cây.
>
> **🧭 Vì sao bước này?** Cây nhỏ đọc được bằng mắt — đó là ưu điểm lớn của cây quyết định. Đề thi giữ kín thay cho tập test.
>
> **📋 Nhiệm vụ:**
> 1. Chia, stratify.
> 2. Mốc: luôn đoán lớp nhiều nhất.
> 3. Cây quyết định `max_depth=3`, vẽ cây.
> 4. Đọc câu hỏi đầu tiên của cây.
>
> **💡 Gợi ý lệnh:**
> - `DecisionTreeClassifier(max_depth=3)` — cây nhỏ, đọc được
> - `plot_tree(...)` — vẽ cây
>
> **✅ Đạt khi:** có mốc, độ chính xác của cây, một câu đọc cây bằng lời.


In [ ]:
# Buoc 2 - code cua nhom



**Nhận xét của nhóm:** …


### ✅ Nộp M1 + M2 (tuần 1)

- [ ] Bảng câu hỏi của nhóm
- [ ] Bước 1 – 2 có nhận xét kèm số, một câu đọc cây
- [ ] *Runtime → Restart and run all* chạy hết không lỗi
- [ ] Link notebook (*Chia sẻ → Bất kỳ ai có đường liên kết · Người xem*) nộp lên Canvas trước 23:59 Thứ 7


---
# TUẦN 2 (buổi 19) — Bảng so sánh 6 model

**Mốc của tuần:** **M3a** Bảng so sánh 6 model  
**Học trước (bài thư viện trên web, 3 chặng bắt buộc + checkpoint):** [Bài 20 Random Forest](https://vuongndlst.github.io/ml-level1/bai20/), [Bài 21 Naïve Bayes](https://vuongndlst.github.io/ml-level1/bai21/), [Bài 22 SVM](https://vuongndlst.github.io/ml-level1/bai22/)


---
## Bước 3 — Bảng so sánh

> **🎯 Mục tiêu:** Huấn luyện 6 model trên cùng train, chấm cùng validation; so độ chính xác, F1, thời gian.
>
> **🧭 Vì sao bước này?** Chỉ so được khi mọi model học cùng train và chấm cùng validation. Một con số chưa đủ: còn thời gian, độ dễ giải thích.
>
> **📋 Nhiệm vụ:**
> 1. Chạy hàm `so_sanh`.
> 2. Đọc bảng: model nào đúng nhất, nhanh nhất?
> 3. Vai Nghiên cứu: mỗi bạn đọc một bài thư viện (20, 21 hoặc 22), giải thích một model cho nhóm.
>
> **💡 Gợi ý lệnh:**
> - `make_pipeline(StandardScaler(), SVC())` — SVM cần cùng thang đo
> - `f1_score(y, du, average="macro")` — F1 trung bình các lớp
>
> **✅ Đạt khi:** có bảng 6 model, một câu so sánh có số.


In [ ]:
# Buoc 3 - code cua nhom



**Nhận xét của nhóm:** …


### ✅ Nộp M3a (tuần 2)

- [ ] Bảng so sánh 6 model + nhận xét
- [ ] *Runtime → Restart and run all* chạy hết không lỗi
- [ ] Link notebook (*Chia sẻ → Bất kỳ ai có đường liên kết · Người xem*) nộp lên Canvas trước 23:59 Thứ 7


---
# TUẦN 3 (buổi 20) — Kiểm định chéo, dò tham số, đề chọn thêm

**Mốc của tuần:** **M3b** Kiểm định chéo, dò siêu tham số, đề chọn thêm  
**Học trước (bài thư viện trên web, 3 chặng bắt buộc + checkpoint):** [Bài 24 Đánh giá cho công bằng](https://vuongndlst.github.io/ml-level1/bai24/), [Bài 25 Chọn model nào?](https://vuongndlst.github.io/ml-level1/bai25/)


---
## Bước 4 — Kiểm định chéo và dò siêu tham số

> **🎯 Mục tiêu:** Đo lại 3 model tốt nhất bằng kiểm định chéo 5 lần; dò C cho SVM.
>
> **🧭 Vì sao bước này?** Một lần chia có thể may hoặc rủi. Kiểm định chéo cho trung bình và độ dao động — so model công bằng hơn.
>
> **📋 Nhiệm vụ:**
> 1. `cross_val_score` cho 3 model, chia 5 phần **có xáo trộn**.
> 2. Đọc trung bình **và** dao động.
> 3. `GridSearchCV` thử 5 giá trị C.
>
> **💡 Gợi ý lệnh:**
> - `StratifiedKFold(5, shuffle=True, random_state=42)` — chia 5 phần, xáo trộn trước
> - `GridSearchCV(m, {"svc__C": [...]}, cv=5)` — dò siêu tham số
>
> **✅ Đạt khi:** có bảng kiểm định chéo và C tốt nhất.


In [ ]:
# Buoc 4 - code cua nhom



**Nhận xét của nhóm:** …


---
## Bước 4b — Đề chọn thêm: nấm hoặc Pokémon

> **🎯 Mục tiêu:** Chạy lại `so_sanh` trên bộ thứ hai; xem model nào thắng, thước đo nào hợp.
>
> **🧭 Vì sao bước này?** Không model nào thắng ở mọi bộ dữ liệu. Chọn model phải theo dữ liệu và theo loại sai mình quan tâm.
>
> **📋 Nhiệm vụ:**
> 1. Chọn **một** bộ: `CHON = "nam"` hoặc `"pokemon"`.
> 2. So mốc và bảng 6 model.
> 3. Với lớp hiếm (nấm độc, Pokémon huyền thoại): đọc cột “bắt được … (%)”.
>
> **💡 Gợi ý lệnh:**
> - `pd.get_dummies(...)` — nấm: mọi cột là chữ
> - `so_sanh(..., lop_quan_tam=...)` — thêm cột bắt được lớp hiếm
>
> **✅ Đạt khi:** trả lời được: model thắng ở hạt đậu có thắng ở bộ này không, thước đo nào nên dùng.


In [ ]:
# Buoc 4b - code cua nhom
CHON = "pokemon"   # hoặc "nam"



**Nhận xét của nhóm:** …


### ✅ Nộp M3b (tuần 3)

- [ ] Bảng kiểm định chéo + C tốt nhất
- [ ] Bước 4b có nhận xét về thước đo
- [ ] *Runtime → Restart and run all* chạy hết không lỗi
- [ ] Link notebook (*Chia sẻ → Bất kỳ ai có đường liên kết · Người xem*) nộp lên Canvas trước 23:59 Thứ 7


---
# TUẦN 4 (buổi 21) — Nộp đề thi, giải thích lựa chọn

**Mốc của tuần:** **M4** Nộp dự đoán đề thi, 1 slide “vì sao chọn model này”  
**Học trước (bài thư viện trên web, 3 chặng bắt buộc + checkpoint):** —


---
## Bước 5 — Chốt model và nộp đề thi

> **🎯 Mục tiêu:** Huấn luyện lại model đã chọn trên toàn bộ dữ liệu có nhãn, dự đoán đề thi.
>
> **🧭 Vì sao bước này?** Đề thi giữ kín là test thật: nhóm không thể “chỉnh cho đẹp” trên nó.
>
> **📋 Nhiệm vụ:**
> 1. Chọn model theo **tiêu chí của nhóm** (điểm kiểm định chéo, dao động, thời gian, dễ giải thích).
> 2. Huấn luyện lại trên toàn bộ `X, y`.
> 3. Dự đoán `de_thi`, tải `du_doan_nhom.csv`, nộp Canvas.
>
> **💡 Gợi ý lệnh:**
> - `pd.DataFrame({"id": ..., "du_doan": ...})` — đúng 2 cột id, du_doan
> - `files.download("du_doan_nhom.csv")` — tải về máy
>
> **✅ Đạt khi:** có file 2 723 dòng, 2 cột; một đoạn giải thích vì sao chọn model này.


In [ ]:
# Buoc 5 - code cua nhom



**Nhận xét của nhóm:** …


### Slide kết quả (M4) — một slide, 4 ô

| Ô | Nội dung |
|---|---|
| Model nhóm chọn | … |
| Tiêu chí | … |
| Một con số | … |
| Bài học từ bộ thứ hai | … |
| Giới hạn | … |

*Làm trên Canva hoặc Google Slides; dán link vào ô nộp Canvas cùng link notebook.*


---
## Bước 6 — Nhật ký AI

Không dùng AI thì ghi *Không dùng*. Kiểm chứng = nhóm tự chạy lại, đối chiếu số, sửa chỗ AI sai.

| Nhóm hỏi AI | AI trả lời / viết | Nhóm kiểm chứng và sửa |
|---|---|---|
| … | … | … |
| … | … | … |
| … | … | … |


---
## Mở rộng — nhóm xong sớm (không bắt buộc)

1. Thêm `GradientBoostingClassifier` vào `cac_model()` — có vượt SVM không?
2. Ma trận nhầm lẫn của model chốt: hai loại hạt nào hay nhầm nhất? Xem ảnh hai loại đó trên mạng — có giống nhau thật không?
3. Pokémon: đổi `class_weight="balanced"` cho SVM, cột “bắt được” thay đổi thế nào?


In [ ]:
# Mo rong - code cua nhom



---
## Trước khi nộp

- [ ] *Runtime → Restart and run all* chạy hết không lỗi
- [ ] Mỗi bước có một ô nhận xét **có số và cỡ nhóm**
- [ ] Không có câu “… **gây ra** …” khi dữ liệu chỉ cho thấy đi cùng nhau
- [ ] Bảng vai trò và nhật ký AI đã điền
- [ ] Link notebook ở chế độ *Người xem*, dán lên Canvas cùng link Canva
